<a href="https://colab.research.google.com/github/R341CUL/python-together/blob/main/10-projects/03_ml_credit_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project · A credit-model scorecard

A compact end-to-end ML deliverable: generate synthetic borrowers, train a
proper pipeline, and print a scorecard (default rate, AUC, full classification
report). This is the fintech credit lesson turned into a reusable mini-project —
the shape of real underwriting analytics. (Synthetic; illustrative only.)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Synthetic borrowers

Income, utilisation, and recent late payments drive a baked-in default
probability; we then sample who defaulted.

In [ ]:
rng = np.random.default_rng(1)
n = 800
income = rng.lognormal(mean=10.6, sigma=0.35, size=n)
utilization = rng.beta(2, 2, size=n)
late_12m = rng.poisson(0.5, size=n)

logit = -2.0 + 2.6 * utilization + 0.55 * late_12m - 0.000035 * income
p = 1 / (1 + np.exp(-logit))
y = rng.binomial(1, p)
X = np.column_stack([income, utilization, late_12m])

## Train a pipeline and print the scorecard

Scale + logistic regression inside a `Pipeline` (ML lesson 5). We judge with AUC
(ranking quality) and a classification report at a chosen decline threshold.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=1
)
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=400)),
]).fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
y_hat = (proba >= 0.35).astype(int)

print("=== CREDIT MODEL SCORECARD ===")
print("portfolio default rate:", round(float(y.mean()), 3))
print("test AUC             :", round(roc_auc_score(y_test, proba), 3))
print()
print(classification_report(y_test, y_hat, digits=3))

## 🏋️ Make it yours

- Swap `LogisticRegression` for `RandomForestClassifier` and compare AUC.
- Sweep the decline threshold from 0.2 to 0.6 and plot approvals vs. defaults
  caught (bring in matplotlib from the data-science track).
- Add a fourth feature (e.g. loan-to-income) and see if AUC improves.

In [ ]:
# experiment here

### What you practised
- An end-to-end supervised-ML pipeline on structured financial data.
- AUC for imbalanced classification; a threshold as a business decision.
- Packaging a lesson's technique into a reusable deliverable.